# 🌍 Audio Language Classifier
**Architecture:** wav2vec2-xls-r-300m (frozen) → mean-pooled 1024-dim embeddings → LightGBM / SVM / LR

**Languages:** Arabic, English, French, Spanish

**Data:** Mozilla Common Voice 17.0 (streamed via HuggingFace — no full download)

---
Run cells **top-to-bottom**. Each cell is checkpointed so you can resume if Colab disconnects.

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 0 — Setup & Installs
# ════════════════════════════════════════════════════════════════
import os, sys, warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/language_classifier'
    !pip install -q transformers datacollective librosa soundfile lightgbm torch torchaudio
else:
    PROJECT_ROOT = os.path.abspath('.')

for d in ['data/raw', 'data/processed', 'data/embeddings', 'outputs', 'models']:
    os.makedirs(f'{PROJECT_ROOT}/{d}', exist_ok=True)

import torch
import numpy as np
import pandas as pd

print(f'Project root : {PROJECT_ROOT}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Configuration
# Edit SAMPLES_PER_LANGUAGE to control dataset size.
# Edit LANGUAGES / MDC_DATASET_IDS to add/remove languages.
# ════════════════════════════════════════════════════════════════

LANGUAGES = [
    ('ar',    'Arabic'),
    ('en',    'English'),
    ('fr',    'French'),
    ('es',    'Spanish'),
]

# ── Mozilla Data Collective dataset IDs ───────────────────────
# Find IDs at: https://datacollective.mozillafoundation.org/datasets?q=common_voice
# The ID is the last segment of the dataset URL.
# Example: .../datasets/cminc35no007no707hql26lzk → "cminc35no007no707hql26lzk"
MDC_DATASET_IDS = {
    'ar':    'cmj8u3os6000tnxxb169x1zdc',  # ← paste Arabic Common Voice dataset ID here
    'en':    'cmihqzerk023co20749miafhq',  # ← paste English Common Voice dataset ID here
    'fr':    'cmj8u48ad0025nxzp2lfaluce',  # ← paste French Common Voice dataset ID here
    'es':    'cmj8u48a70021nxzpqvs3sisu',  # ← paste Spanish Common Voice dataset ID here
}

# Set to 'mdc' to download from Mozilla Data Collective (recommended)
# Set to 'streaming' for legacy HuggingFace streaming (may not work)
DATA_SOURCE = 'mdc'

LANG_CODES = [c for c, _ in LANGUAGES]
LANG_NAMES = [n for _, n in LANGUAGES]
NUM_CLASSES = len(LANGUAGES)

CODE_TO_LABEL = {c: i for i, (c, _) in enumerate(LANGUAGES)}
LABEL_TO_CODE = {i: c for c, i in CODE_TO_LABEL.items()}
LABEL_TO_NAME = {i: n for i, (_, n) in enumerate(LANGUAGES)}

# Audio
SAMPLE_RATE     = 16_000
SEGMENT_SEC     = 3.0
TARGET_SAMPLES  = int(SAMPLE_RATE * SEGMENT_SEC)  # 48000

# wav2vec2
WAV2VEC_MODEL   = 'facebook/wav2vec2-xls-r-300m'
EMBEDDING_DIM   = 1024
BATCH_SIZE      = 16
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'

# Data
SAMPLES_PER_LANGUAGE = 500   # ← change this (50 for quick test, 500+ for real)
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# SVM cap
MAX_SVM_SAMPLES = 5000

print(f'Data source  : {DATA_SOURCE}')
print(f'Languages    : {NUM_CLASSES}')
print(f'Samples/lang : {SAMPLES_PER_LANGUAGE}')
print(f'Total samples: ~{NUM_CLASSES * SAMPLES_PER_LANGUAGE}')
print(f'Device       : {DEVICE}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — Download / Stream data
#
# Option A (recommended): Mozilla Data Collective
#   1. Sign up at https://datacollective.mozillafoundation.org/auth/signup
#   2. Create API key at Profile > Credentials
#   3. Accept dataset terms on the website
#   4. Fill in MDC_DATASET_IDS in Cell 1
#   5. Set your API key below
#
# Option B (legacy fallback): HuggingFace streaming
#   Set DATA_SOURCE = 'streaming' in Cell 1
# ════════════════════════════════════════════════════════════════
import glob, tarfile, zipfile
from tqdm.auto import tqdm

MANIFEST_PATH = f'{PROJECT_ROOT}/outputs/manifest.csv'

# Try to load an existing manifest
manifest_df = None
if os.path.exists(MANIFEST_PATH):
    try:
        _df = pd.read_csv(MANIFEST_PATH)
        required_cols = {'language', 'sample_id', 'npy_path'}
        if len(_df) > 0 and required_cols.issubset(_df.columns):
            # Filter to only current languages
            _df = _df[_df['language'].isin(LANG_NAMES)].reset_index(drop=True)
            if len(_df) > 0:
                manifest_df = _df
                print(f'Manifest already exists: {len(manifest_df)} samples')
                print(manifest_df['language'].value_counts().to_string())
            else:
                print('Manifest has no samples for current languages — will re-download.')
                os.remove(MANIFEST_PATH)
        else:
            print(f'Manifest exists but is empty/invalid (cols={list(_df.columns)}, rows={len(_df)}) — will re-download.')
            os.remove(MANIFEST_PATH)
    except Exception as e:
        print(f'Manifest file is corrupt ({e}) — deleting and re-downloading.')
        os.remove(MANIFEST_PATH)

if manifest_df is None:

    if DATA_SOURCE == 'mdc':
        # ── Mozilla Data Collective download ──────────────────────
        import getpass
        if not os.environ.get('MDC_API_KEY'):
            os.environ['MDC_API_KEY'] = getpass.getpass('Enter your MDC API key: ')

        # Set download path so all datasets land under our project
        MDC_RAW_DIR = f'{PROJECT_ROOT}/data/raw'
        os.environ['MDC_DOWNLOAD_PATH'] = MDC_RAW_DIR

        from datacollective import save_dataset_to_disk

        records = []

        for lang_code, lang_name in LANGUAGES:
            dataset_id = MDC_DATASET_IDS.get(lang_code, '')
            if not dataset_id:
                print(f'\n  ⚠ Skipping {lang_name} — no MDC dataset ID configured.')
                print(f'    Find it at: https://datacollective.mozillafoundation.org/datasets?q=common_voice')
                continue

            print(f"\n{'='*50}")
            print(f'  Downloading {lang_name} ({lang_code}) from MDC...')
            print(f'  Dataset ID: {dataset_id}')
            print(f"{'='*50}")

            try:
                # save_dataset_to_disk downloads to MDC_DOWNLOAD_PATH
                dataset_result = save_dataset_to_disk(dataset_id)
                print(f'  Download complete: {dataset_result}')

                # Determine where the files landed
                lang_raw_dir = MDC_RAW_DIR  # files are under MDC_DOWNLOAD_PATH

                # If result is a path string, use its parent directory
                if isinstance(dataset_result, str) and os.path.exists(dataset_result):
                    if os.path.isfile(dataset_result):
                        lang_raw_dir = os.path.dirname(dataset_result)
                        # Extract archives
                        if dataset_result.endswith(('.tar.gz', '.tgz')):
                            print(f'  Extracting tar.gz...')
                            with tarfile.open(dataset_result, 'r:gz') as tar:
                                tar.extractall(path=lang_raw_dir)
                        elif dataset_result.endswith('.zip'):
                            print(f'  Extracting zip...')
                            with zipfile.ZipFile(dataset_result, 'r') as zf:
                                zf.extractall(path=lang_raw_dir)
                    else:
                        lang_raw_dir = dataset_result

                # Find all audio files recursively
                audio_files = sorted(
                    glob.glob(os.path.join(lang_raw_dir, '**', '*.mp3'), recursive=True)
                    + glob.glob(os.path.join(lang_raw_dir, '**', '*.wav'), recursive=True)
                    + glob.glob(os.path.join(lang_raw_dir, '**', '*.flac'), recursive=True)
                    + glob.glob(os.path.join(lang_raw_dir, '**', '*.opus'), recursive=True)
                )

                # Also check the default MDC path ~/.mozdata/datasets
                if not audio_files:
                    default_mdc = os.path.expanduser('~/.mozdata/datasets')
                    audio_files = sorted(
                        glob.glob(os.path.join(default_mdc, '**', '*.mp3'), recursive=True)
                        + glob.glob(os.path.join(default_mdc, '**', '*.wav'), recursive=True)
                        + glob.glob(os.path.join(default_mdc, '**', '*.flac'), recursive=True)
                        + glob.glob(os.path.join(default_mdc, '**', '*.opus'), recursive=True)
                    )

                if not audio_files:
                    print(f'  ⚠ No audio files found. Check {lang_raw_dir}')
                    print(f'    Dataset result was: {dataset_result} (type: {type(dataset_result).__name__})')
                    # List what's actually there
                    for root, dirs, files in os.walk(lang_raw_dir):
                        for f in files[:5]:
                            print(f'      {os.path.join(root, f)}')
                        if len(files) > 5:
                            print(f'      ... and {len(files)-5} more files')
                        break
                    continue

                print(f'  Found {len(audio_files)} audio files')
                import librosa

                count = 0
                for fpath in tqdm(audio_files, desc=lang_name):
                    if count >= SAMPLES_PER_LANGUAGE:
                        break

                    try:
                        y, _ = librosa.load(fpath, sr=SAMPLE_RATE, mono=True)
                        y = y.astype(np.float32)
                    except Exception:
                        continue

                    if len(y) < SAMPLE_RATE * 0.5:
                        continue
                    if np.sqrt(np.mean(y**2)) < 1e-5:
                        continue

                    if len(y) > TARGET_SAMPLES:
                        y = y[:TARGET_SAMPLES]
                    elif len(y) < TARGET_SAMPLES:
                        y = np.pad(y, (0, TARGET_SAMPLES - len(y)))

                    sample_id = f'{lang_code}_{count:05d}'
                    npy_path  = f'{PROJECT_ROOT}/data/processed/{sample_id}.npy'
                    np.save(npy_path, y)

                    records.append({
                        'sample_id': sample_id, 'language_code': lang_code,
                        'language': lang_name, 'npy_path': npy_path,
                        'label': CODE_TO_LABEL[lang_code],
                    })
                    count += 1

                print(f'  ✅ {lang_name}: {count} samples processed')

            except Exception as e:
                import traceback
                print(f'  ❌ Failed: {e}')
                traceback.print_exc()
                continue

        manifest_df = pd.DataFrame(records)
        manifest_df.to_csv(MANIFEST_PATH, index=False)
        print(f'\nManifest saved: {len(manifest_df)} total samples')
        if len(manifest_df) > 0:
            print(manifest_df['language'].value_counts().to_string())

    else:
        # ── Legacy: HuggingFace streaming fallback ────────────────
        from datasets import load_dataset, Audio

        print('⚠ Using legacy HuggingFace streaming.')
        print('  Common Voice has moved to: https://datacollective.mozillafoundation.org')
        print('  Set DATA_SOURCE = "mdc" in Cell 1 to use the new source.\n')

        records = []

        for lang_code, lang_name in LANGUAGES:
            print(f"\n{'='*50}")
            print(f'  Loading {lang_name} ({lang_code})...')
            print(f"{'='*50}")

            try:
                ds = load_dataset(
                    'mozilla-foundation/common_voice_17_0',
                    lang_code, split='train', streaming=True, trust_remote_code=True,
                )
                ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))

                count = 0
                for sample in tqdm(ds, total=SAMPLES_PER_LANGUAGE, desc=lang_name):
                    if count >= SAMPLES_PER_LANGUAGE:
                        break

                    audio = sample['audio']
                    y = np.array(audio['array'], dtype=np.float32)

                    if len(y) < SAMPLE_RATE * 0.5:
                        continue
                    if np.sqrt(np.mean(y**2)) < 1e-5:
                        continue

                    if len(y) > TARGET_SAMPLES:
                        y = y[:TARGET_SAMPLES]
                    elif len(y) < TARGET_SAMPLES:
                        y = np.pad(y, (0, TARGET_SAMPLES - len(y)))

                    sample_id = f'{lang_code}_{count:05d}'
                    npy_path  = f'{PROJECT_ROOT}/data/processed/{sample_id}.npy'
                    np.save(npy_path, y)

                    records.append({
                        'sample_id': sample_id, 'language_code': lang_code,
                        'language': lang_name, 'npy_path': npy_path,
                        'label': CODE_TO_LABEL[lang_code],
                    })
                    count += 1

                print(f'  ✅ {lang_name}: {count} samples saved')
            except Exception as e:
                print(f'  ❌ Failed to load {lang_name}: {e}')
                continue

        manifest_df = pd.DataFrame(records)
        manifest_df.to_csv(MANIFEST_PATH, index=False)
        print(f'\nManifest saved: {len(manifest_df)} total samples')
        if len(manifest_df) > 0:
            print(manifest_df['language'].value_counts().to_string())

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — Train / Val / Test Split
# ════════════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split

df = manifest_df.copy()
print(f'Manifest columns: {list(df.columns)}  rows: {len(df)}')

if len(df) == 0 or len(df.columns) == 0:
    raise RuntimeError(
        '❌ Manifest is empty! Delete the cached manifest and re-run Cell 3 (Download):\n'
        f'    import os; os.remove("{PROJECT_ROOT}/outputs/manifest.csv")\n'
        '    Then re-run Cell 3.'
    )

# Filter to only current languages (drop old ones like zh-CN)
if 'language_code' in df.columns:
    df = df[df['language_code'].isin(LANG_CODES)].reset_index(drop=True)
elif 'language' in df.columns:
    df = df[df['language'].isin(LANG_NAMES)].reset_index(drop=True)
print(f'After filtering to {LANG_CODES}: {len(df)} samples')

if len(df) == 0:
    raise RuntimeError(
        '❌ No samples left after filtering to current languages!\n'
        '    Delete the old manifest and re-run Cell 3:\n'
        f'    import os; os.remove("{PROJECT_ROOT}/outputs/manifest.csv")'
    )

# Reconstruct 'label' column if missing (old cached manifest)
if 'label' not in df.columns:
    if 'language_code' in df.columns:
        df['label'] = df['language_code'].map(CODE_TO_LABEL)
    elif 'language' in df.columns:
        name_to_label = {n: i for i, (_, n) in enumerate(LANGUAGES)}
        df['label'] = df['language'].map(name_to_label)
    else:
        raise ValueError(f'Cannot reconstruct labels — manifest has no language_code or language column. Columns: {list(df.columns)}')
    df = df.dropna(subset=['label']).reset_index(drop=True)
    df['label'] = df['label'].astype(int)
    print(f'Reconstructed label column: {df["label"].value_counts().to_dict()}')

train_df, temp_df = train_test_split(
    df, test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df['label'], random_state=42,
)

val_df, test_df = train_test_split(
    temp_df, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=temp_df['label'], random_state=42,
)

print(f'Train : {len(train_df)}')
print(f'Val   : {len(val_df)}')
print(f'Test  : {len(test_df)}')
print(f'\nTrain distribution:')
print(train_df['language'].value_counts().to_string())

train_df.to_csv(f'{PROJECT_ROOT}/outputs/train_split.csv', index=False)
val_df.to_csv(f'{PROJECT_ROOT}/outputs/val_split.csv', index=False)
test_df.to_csv(f'{PROJECT_ROOT}/outputs/test_split.csv', index=False)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — wav2vec2 Embedding Extraction
# Model: facebook/wav2vec2-xls-r-300m (multilingual, frozen)
# Output: (N, 1024) embeddings via mean pooling
# ════════════════════════════════════════════════════════════════
from transformers import Wav2Vec2Model, Wav2Vec2Processor

EMBED_DIR   = f'{PROJECT_ROOT}/data/embeddings'
EMBED_FILE  = f'{EMBED_DIR}/embeddings.npy'
SEGIDS_FILE = f'{EMBED_DIR}/seg_ids.npy'

# Combine all splits for extraction
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

if os.path.exists(EMBED_FILE) and os.path.exists(SEGIDS_FILE):
    print('Embeddings already exist — loading from disk...')
    X_all       = np.load(EMBED_FILE)
    seg_ids_all = np.load(SEGIDS_FILE, allow_pickle=True)
    print(f'  Shape: {X_all.shape}')
else:
    print(f'Loading {WAV2VEC_MODEL}...')
    processor = Wav2Vec2Processor.from_pretrained(WAV2VEC_MODEL)
    model     = Wav2Vec2Model.from_pretrained(WAV2VEC_MODEL).to(DEVICE)
    model.eval()
    print(f'  Model loaded: {sum(p.numel() for p in model.parameters()):,} params')

    def extract_batch(paths):
        waveforms = [np.load(p).astype(np.float32) for p in paths]
        inputs = processor(
            waveforms, sampling_rate=SAMPLE_RATE,
            return_tensors='pt', padding=True,
        )
        input_values = inputs.input_values.to(DEVICE)
        with torch.no_grad():
            outputs = model(input_values)
            embeddings = outputs.last_hidden_state.mean(dim=1)
        return embeddings.cpu().numpy()

    all_paths   = all_df['npy_path'].tolist()
    all_seg_ids = all_df['sample_id'].tolist()
    all_embeds  = []

    CKPT_EMBED = f'{EMBED_DIR}/embeddings_partial.npy'
    start_idx  = 0
    if os.path.exists(CKPT_EMBED):
        prev = np.load(CKPT_EMBED)
        all_embeds.append(prev)
        start_idx = prev.shape[0]
        print(f'  Resuming from sample {start_idx}')

    remaining = all_paths[start_idx:]

    for i in tqdm(range(0, len(remaining), BATCH_SIZE), desc='Extracting embeddings'):
        batch_paths = remaining[i : i + BATCH_SIZE]
        try:
            embeds = extract_batch(batch_paths)
            all_embeds.append(embeds)
        except RuntimeError as e:
            if 'CUDA out of memory' in str(e):
                torch.cuda.empty_cache()
                print(f'\nOOM at batch {i} — reduce BATCH_SIZE')
                raise
            print(f'  ⚠ Batch {i} error: {e}')
            all_embeds.append(np.zeros((len(batch_paths), EMBEDDING_DIM)))

        # Checkpoint every 50 batches
        if (i // BATCH_SIZE + 1) % 50 == 0:
            partial = np.vstack(all_embeds)
            np.save(CKPT_EMBED, partial)
            print(f'  Checkpoint: {partial.shape[0]} embeddings')

    X_all       = np.vstack(all_embeds)
    seg_ids_all = np.array(all_seg_ids[:X_all.shape[0]])

    np.save(EMBED_FILE, X_all)
    np.save(SEGIDS_FILE, seg_ids_all)
    if os.path.exists(CKPT_EMBED):
        os.remove(CKPT_EMBED)

    print(f'\nEmbeddings: {X_all.shape}')
    print(f'Saved to {EMBED_FILE}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — Build Feature Matrices & Scale
# ════════════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler
import joblib

sid_to_idx = {sid: i for i, sid in enumerate(seg_ids_all)}

def build_matrix(split_df):
    X_list, y_list = [], []
    for _, row in split_df.iterrows():
        sid = row['sample_id']
        if sid in sid_to_idx:
            X_list.append(X_all[sid_to_idx[sid]])
            y_list.append(row['label'])
    return np.array(X_list), np.array(y_list)

X_train, y_train = build_matrix(train_df)
X_val,   y_val   = build_matrix(val_df)
X_test,  y_test  = build_matrix(test_df)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val  : {X_val.shape}    y_val  : {y_val.shape}')
print(f'X_test : {X_test.shape}   y_test : {y_test.shape}')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, f'{PROJECT_ROOT}/models/scaler.joblib')

np.savez(
    f'{PROJECT_ROOT}/outputs/features.npz',
    X_train=X_train_sc, y_train=y_train,
    X_val=X_val_sc, y_val=y_val,
    X_test=X_test_sc, y_test=y_test,
)
print('Features saved ✅')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — Train Classifiers
#   1. LightGBM   (primary)
#   2. SVM RBF    (subsampled)
#   3. Logistic Regression (baseline)
# ════════════════════════════════════════════════════════════════
import lightgbm as lgb
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# ── LightGBM ─────────────────────────────────────────────────
print('='*55)
print('  Training LightGBM...')
print('='*55)

lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    max_depth=8, class_weight='balanced', random_state=42,
    n_jobs=-1, verbose=-1,
)
lgb_model.fit(
    X_train_sc, y_train,
    eval_set=[(X_val_sc, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=True)],
)

y_pred_lgb = lgb_model.predict(X_test_sc)
lgb_acc = accuracy_score(y_test, y_pred_lgb)
lgb_f1  = f1_score(y_test, y_pred_lgb, average='macro')
print(f'\nLightGBM — Accuracy: {lgb_acc*100:.2f}%  Macro-F1: {lgb_f1:.4f}')

# ── SVM ───────────────────────────────────────────────────────
print(f"\n{'='*55}")
print('  Training SVM (RBF)...')
print('='*55)

if X_train_sc.shape[0] > MAX_SVM_SAMPLES:
    rng = np.random.RandomState(42)
    svm_idx = rng.choice(X_train_sc.shape[0], MAX_SVM_SAMPLES, replace=False)
    X_train_svm, y_train_svm = X_train_sc[svm_idx], y_train[svm_idx]
    print(f'  Subsampled to {MAX_SVM_SAMPLES}')
else:
    X_train_svm, y_train_svm = X_train_sc, y_train

svm_model = SVC(
    kernel='rbf', C=10, gamma='scale', class_weight='balanced',
    probability=True, random_state=42, decision_function_shape='ovr',
)
svm_model.fit(X_train_svm, y_train_svm)

y_pred_svm = svm_model.predict(X_test_sc)
svm_acc = accuracy_score(y_test, y_pred_svm)
svm_f1  = f1_score(y_test, y_pred_svm, average='macro')
print(f'SVM — Accuracy: {svm_acc*100:.2f}%  Macro-F1: {svm_f1:.4f}')

# ── Logistic Regression ───────────────────────────────────────
print(f"\n{'='*55}")
print('  Training Logistic Regression...')
print('='*55)

lr_model = LogisticRegression(
    C=1.0, class_weight='balanced', max_iter=2000,
    multi_class='multinomial', random_state=42, n_jobs=-1,
)
lr_model.fit(X_train_sc, y_train)

y_pred_lr = lr_model.predict(X_test_sc)
lr_acc = accuracy_score(y_test, y_pred_lr)
lr_f1  = f1_score(y_test, y_pred_lr, average='macro')
print(f'LR — Accuracy: {lr_acc*100:.2f}%  Macro-F1: {lr_f1:.4f}')

# ── Summary & pick best ───────────────────────────────────────
print(f"\n{'='*55}")
print('  MODEL COMPARISON')
print(f"{'='*55}")
print(f"  {'Model':<25} {'Accuracy':>10} {'Macro-F1':>10}")
print(f"  {'-'*45}")
print(f"  {'LightGBM':<25} {lgb_acc*100:>9.2f}% {lgb_f1:>10.4f}")
print(f"  {'SVM (RBF)':<25} {svm_acc*100:>9.2f}% {svm_f1:>10.4f}")
print(f"  {'Logistic Regression':<25} {lr_acc*100:>9.2f}% {lr_f1:>10.4f}")

best_name, best_model, best_preds = max(
    [('LightGBM', lgb_model, y_pred_lgb),
     ('SVM',      svm_model, y_pred_svm),
     ('LR',       lr_model,  y_pred_lr)],
    key=lambda x: f1_score(y_test, x[2], average='macro'),
)
print(f'\n🏆 Best model: {best_name}')

# Save all models
joblib.dump(lgb_model,  f'{PROJECT_ROOT}/models/lgb_model.joblib')
joblib.dump(svm_model,  f'{PROJECT_ROOT}/models/svm_model.joblib')
joblib.dump(lr_model,   f'{PROJECT_ROOT}/models/lr_model.joblib')
joblib.dump(best_model, f'{PROJECT_ROOT}/models/best_model.joblib')
print('All models saved ✅')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — Detailed Evaluation
# ════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

target_names = [LABEL_TO_NAME[i] for i in range(NUM_CLASSES)]

best_preds_final = best_model.predict(X_test_sc)

print(f"\n{'='*55}")
print(f'  DETAILED EVALUATION — {best_name}')
print(f"{'='*55}")
print('\nClassification Report:')
print(classification_report(y_test, best_preds_final, target_names=target_names))

# Confusion matrix
cm = confusion_matrix(y_test, best_preds_final)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=target_names, yticklabels=target_names, ax=ax,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
overall_acc = accuracy_score(y_test, best_preds_final)
overall_f1  = f1_score(y_test, best_preds_final, average='macro')
ax.set_title(f'Language Classification — {best_name}\n'
             f'Accuracy: {overall_acc*100:.1f}%  Macro-F1: {overall_f1:.4f}')
plt.tight_layout()
fig.savefig(f'{PROJECT_ROOT}/outputs/confusion_matrix.png', dpi=150)
plt.show()
print(f'Confusion matrix saved')

# Per-language accuracy
print('\nPer-language accuracy:')
for i, name in LABEL_TO_NAME.items():
    mask = y_test == i
    if mask.sum() == 0:
        continue
    acc_i = (best_preds_final[mask] == i).mean()
    print(f'  {name:<15} {acc_i*100:>6.1f}%  (n={mask.sum()})')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8 — Inference Function
# Use this to predict the language of any audio file.
# ════════════════════════════════════════════════════════════════
import librosa

def predict_language(audio_path):
    """
    Predict the language of an audio file.

    Returns dict with: language, language_code, confidence, all_probs
    """
    # Load & preprocess
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    if len(y) > TARGET_SAMPLES:
        y = y[:TARGET_SAMPLES]
    elif len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)))

    # Extract embedding
    inputs = processor(
        [y], sampling_rate=SAMPLE_RATE,
        return_tensors='pt', padding=True,
    )
    input_values = inputs.input_values.to(DEVICE)
    with torch.no_grad():
        outputs = model(input_values)
        embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

    # Scale & predict
    embedding_sc = scaler.transform(embedding)
    pred_label = best_model.predict(embedding_sc)[0]

    if hasattr(best_model, 'predict_proba'):
        probs = best_model.predict_proba(embedding_sc)[0]
    else:
        probs = np.zeros(NUM_CLASSES)
        probs[pred_label] = 1.0

    return {
        'language':      LABEL_TO_NAME[pred_label],
        'language_code': LABEL_TO_CODE[pred_label],
        'confidence':    float(probs[pred_label]),
        'all_probs':     {LABEL_TO_NAME[i]: float(p) for i, p in enumerate(probs)},
    }


print('✅ predict_language() ready!')
print()
print('Usage:')
print('  result = predict_language(\'path/to/audio.wav\')')
print('  print(result["language"], result["confidence"])')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 9 — Try it! Upload an audio file and predict.
# ════════════════════════════════════════════════════════════════

if IN_COLAB:
    from google.colab import files
    print('Upload an audio file (.wav, .mp3, .flac):')
    uploaded = files.upload()

    for fname in uploaded:
        print(f'\n--- {fname} ---')
        result = predict_language(fname)
        print(f"  Language   : {result['language']}")
        print(f"  Confidence : {result['confidence']*100:.1f}%")
        print(f"  All probs  :")
        for lang, prob in sorted(result['all_probs'].items(), key=lambda x: -x[1]):
            bar = '█' * int(prob * 30)
            print(f"    {lang:<12} {prob*100:5.1f}% {bar}")
else:
    print('Not in Colab. Use predict_language("path/to/file.wav") directly.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 10 — Save Outputs & Report
# ════════════════════════════════════════════════════════════════
from datetime import date

# Predictions CSV
test_results = pd.DataFrame({
    'sample_id':     test_df['sample_id'].values[:len(y_test)],
    'true_label':    y_test,
    'true_language': [LABEL_TO_NAME[l] for l in y_test],
    'pred_label':    best_preds_final,
    'pred_language': [LABEL_TO_NAME[l] for l in best_preds_final],
    'correct':       (y_test == best_preds_final),
})

if hasattr(best_model, 'predict_proba'):
    probs_all = best_model.predict_proba(X_test_sc)
    test_results['confidence'] = probs_all.max(axis=1).round(4)

test_results.to_csv(f'{PROJECT_ROOT}/outputs/predictions.csv', index=False)

# Metrics
metrics = {
    'model': best_name, 'accuracy': overall_acc,
    'macro_f1': overall_f1, 'num_classes': NUM_CLASSES,
    'n_train': len(y_train), 'n_test': len(y_test),
}
pd.DataFrame([metrics]).to_csv(f'{PROJECT_ROOT}/outputs/metrics.csv', index=False)

# Report
report = f"""# Language Classification Report
**Date:** {date.today().strftime('%B %d, %Y')}

## Configuration
- Languages: {', '.join(LANG_NAMES)}
- Samples per language: {SAMPLES_PER_LANGUAGE}
- Feature extractor: {WAV2VEC_MODEL} (frozen)
- Best classifier: {best_name}

## Results
- **Accuracy:** {overall_acc*100:.2f}%
- **Macro F1:** {overall_f1:.4f}
- Test set size: {len(y_test)} samples

## Model Comparison
| Model | Accuracy | Macro-F1 |
|-------|----------|----------|
| LightGBM | {lgb_acc*100:.2f}% | {lgb_f1:.4f} |
| SVM (RBF) | {svm_acc*100:.2f}% | {svm_f1:.4f} |
| Logistic Regression | {lr_acc*100:.2f}% | {lr_f1:.4f} |
"""

with open(f'{PROJECT_ROOT}/outputs/report.md', 'w') as f:
    f.write(report)

print(f"{'='*50}")
print('  ALL OUTPUTS SAVED ✅')
print(f"{'='*50}")
print(f'  {PROJECT_ROOT}/outputs/predictions.csv')
print(f'  {PROJECT_ROOT}/outputs/confusion_matrix.png')
print(f'  {PROJECT_ROOT}/outputs/metrics.csv')
print(f'  {PROJECT_ROOT}/outputs/report.md')
print(f'  {PROJECT_ROOT}/models/best_model.joblib')
print(f"{'='*50}")